# Mô hình dự đoán `is_canceled` — LightGBM **v2.2**

**Hướng (idea-refine):** V1+V2 — FE booking-time bổ sung + calibration + tối ưu ngưỡng (Recall ≥ 0,85).

| Thành phần | v2.1 | **v2.2** |
|------------|------|----------|
| Features | FE v1.2 + `arrival_season` | + `meal`, `reserved_room_type`, `booking_changes`, `is_repeated_guest`, … |
| Prob | raw LightGBM | **Isotonic calibration** (`CalibratedClassifierCV`) |
| Ngưỡng | 0,28 / policy 0,51 | **Chọn trên train-val** (min FP ‖ Recall ≥ 0,85) |
| Params | `best_params_v2_1.json` | Warm-start cùng params + artifact `best_params_v2_2.json` |

**Claims:** xem `refine-logs/EXPERIMENT_PLAN.md`  
**Chống leakage:** không dùng `reservation_status*`, `revenue`, `Occupancy_Rate`, `RevPAR`, `assigned_room_type`.

Script đầy đủ: `_run_v2_2.py` (khuyến nghị chạy 1 lần để sinh artifact + báo cáo).


## 0. Cấu hình & chạy pipeline thí nghiệm (M0–M3)

In [ ]:
from pathlib import Path
import os
import sys

# Resolve roots
_cwd = Path.cwd().resolve()
if (_cwd / "data" / "hotel_bookings_v5.csv").exists():
    ROOT = _cwd
elif (_cwd.parents[1] / "data" / "hotel_bookings_v5.csv").exists():
    ROOT = _cwd.parents[1]
else:
    ROOT = Path("..").resolve().parent if Path("data").exists() is False else _cwd

MODEL_DIR = ROOT / "models" / "Cancellation Predict Model v2"
SCRIPT = MODEL_DIR / "_run_v2_2.py"

# Flags: SHAP=1, W&B offline optional
os.environ.setdefault("RUN_SHAP", "1")
os.environ.setdefault("USE_WANDB", "0")  # đặt 1 nếu đã wandb login
os.environ.setdefault("WANDB_MODE", "offline")

print("ROOT:", ROOT)
print("Script:", SCRIPT, "exists=", SCRIPT.exists())


In [ ]:
# Chạy toàn bộ experiment plan (R001–R004). Có thể mất ~5–20 phút.
# Nếu %run lỗi path, chạy từ terminal:
#   python "_run_v2_2.py"
import runpy
runpy.run_path(str(SCRIPT), run_name="__main__")


## 1. Đọc kết quả (`analyze-results`)

Sau khi script xong, artifact nằm ở `artifacts/`:
- `v2_2_comparison.csv`
- `v2_2_ablation.csv`
- `v2_2_stats.json`
- `threshold_policy_v2_2.json`
- `best_params_v2_2.json`

Báo cáo: `reports/09_cancellation_model_v2_2.md`


In [ ]:
import json
import pandas as pd

ART = MODEL_DIR / "artifacts"
compare = pd.read_csv(ART / "v2_2_comparison.csv")
ablation = pd.read_csv(ART / "v2_2_ablation.csv")
stats = json.loads((ART / "v2_2_stats.json").read_text(encoding="utf-8"))
policy = json.loads((ART / "threshold_policy_v2_2.json").read_text(encoding="utf-8"))

display(compare.round(4))
display(ablation.round(4))
print("Decision:", stats.get("decision"))
print("Threshold:", policy["thresholds"])
print("Test metrics:", policy["test_metrics"])


## 2. Biểu đồ báo cáo (giống v2.1)

In [ ]:
from IPython.display import Image, Markdown, display

FIG = ROOT / "reports" / "figures" / "09_v2_2"
chart_titles = {
    "chart_01.png": "Confusion Matrix & ROC",
    "chart_02.png": "Phân phối P(hủy)",
    "chart_03.png": "Feature Importance (gain) Top 20",
    "chart_04.png": "SHAP mean |SHAP| Top 15",
    "chart_05.png": "SHAP Beeswarm",
    "chart_06.png": "So sánh FP / Recall / Precision",
    "chart_07.png": "Confusion v2.1@0.51 vs v2.2",
    "chart_08.png": "Calibration curve",
    "chart_09.png": "FP–Recall theo ngưỡng",
    "chart_10.png": "SHAP Waterfall",
}
for name, title in chart_titles.items():
    path = FIG / name
    display(Markdown(f"### {title}"))
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Thiếu:", path)

shap_json = ART / "v2_2_shap_top20.json"
if shap_json.exists():
    display(Markdown("### SHAP top 20 (bảng)"))
    display(pd.DataFrame(json.loads(shap_json.read_text(encoding="utf-8"))))


## 3. Quyết định ship (ab-test / statistical-analysis)

- Primary: FP dưới ràng buộc Recall ≥ 0,85
- Guardrail: CV ROC-AUC không giảm so v2.1
- McNemar + bootstrap ΔAUC trong `v2_2_stats.json`

Xem recommendation trong báo cáo markdown.
